In [ ]:
# ============================================================
# INTEGRATION CSV → Ventes_fonciere (VERSION FINALE)
# Ordre : COMMUNE → ADRESSE → NATURE_CULTURE → NATURE_CULTURE_SPECIALE
#       → BIEN (4 types) → PARCELLE → MUTATION → MUTATION_BIEN
#       → BIEN_PARCELLE → PARCELLE_NAT_CULT → PARCELLE_NAT_CULT_SPE
# ============================================================
import polars as pl
import pyodbc
import gc
import os
os.environ["POLARS_MAX_THREADS"] = "2"

# CONNEXION
conn = pyodbc.connect(
    "DRIVER={ODBC Driver 17 for SQL Server};"
    "Server=DESKTOP-PDTQGH1\\SQLEXPRESS;"
    "Database=Ventes_fonciere;"
    "Trusted_Connection=yes;"
    "TrustServerCertificate=yes;"
)
print("✅ Connexion établie")

CSV_PATH = r"C:\Users\utilisateur\Desktop\SAE Outil décisionnel, Modélisation statistique et Intelligence Artificielle\DATA\full_data.csv"

# ============================================================
# HELPERS
# ============================================================
def bulk_insert(conn, table_name, df, chunk_size=10_000):
    n = len(df)
    if n == 0:
        print(f"  ⚠️  {table_name} : aucune ligne à insérer")
        return
    cols = df.columns
    placeholders = ",".join(["?" for _ in cols])
    sql = f"INSERT INTO {table_name} ({','.join(cols)}) VALUES ({placeholders})"
    cursor = conn.cursor()
    for start in range(0, n, chunk_size):
        stop = min(start + chunk_size, n)
        chunk = df[start:stop].rows()
        cursor.executemany(sql, chunk)
        conn.commit()
        print(f"\r  {stop:,} / {n:,} lignes insérées dans {table_name}", end="", flush=True)
    cursor.close()
    print()

def load_table(conn, query, schema):
    cursor = conn.cursor()
    cursor.execute(query)
    rows = cursor.fetchall()
    cursor.close()
    if not rows:
        return pl.DataFrame(schema=schema)
    col_names = list(schema.keys())
    columns = {col_names[i]: [row[i] for row in rows] for i in range(len(col_names))}
    return pl.DataFrame(columns).cast(schema)

def get_existing_keys(conn, table_name, key_column):
    cursor = conn.cursor()
    cursor.execute(f"SELECT {key_column} FROM {table_name}")
    keys = set(row[0] for row in cursor.fetchall())
    cursor.close()
    return keys

def count_rows(conn, table_name):
    cursor = conn.cursor()
    cursor.execute(f"SELECT COUNT(*) FROM {table_name}")
    count = cursor.fetchone()[0]
    cursor.close()
    return count

def normalize_cols(df, cols):
    """Normalise les colonnes texte : strip + majuscules + NULL → ''
    CLÉ DU SUCCÈS : permet de matcher 9.7M parcelles avec leurs adresses"""
    return df.with_columns([
        pl.col(c).str.strip_chars().str.to_uppercase().fill_null("")
        for c in cols
    ])

# ============================================================
# LECTURE CSV
# ============================================================
print("\n📂 Lecture du CSV complet en mémoire...")
data = (
    pl.scan_csv(
        CSV_PATH,
        separator=";",
        infer_schema=False,
        null_values=["", "nan", "NaN", "NULL"],
        ignore_errors=True,
        truncate_ragged_lines=True,
    )
    .collect()
)
print(f"✅ CSV chargé : {len(data):,} lignes")

data = data.with_columns([
    pl.col("valeur_fonciere").str.replace(",", ".").cast(pl.Float64, strict=False),
    pl.col("surface_reelle_bati").str.replace(",", ".").cast(pl.Float64, strict=False),
    pl.col("nombre_pieces_principales").str.replace(",", ".").cast(pl.Int32, strict=False),
    pl.col("longitude").str.replace(",", ".").cast(pl.Float64, strict=False),
    pl.col("latitude").str.replace(",", ".").cast(pl.Float64, strict=False),
    pl.col("lot1_surface_carrez").str.replace(",", ".").cast(pl.Float64, strict=False),
    pl.col("lot2_surface_carrez").str.replace(",", ".").cast(pl.Float64, strict=False),
    pl.col("lot3_surface_carrez").str.replace(",", ".").cast(pl.Float64, strict=False),
    pl.col("lot4_surface_carrez").str.replace(",", ".").cast(pl.Float64, strict=False),
    pl.col("lot5_surface_carrez").str.replace(",", ".").cast(pl.Float64, strict=False),
    pl.col("surface_terrain").str.replace(",", ".").cast(pl.Float64, strict=False),
    pl.when(pl.col("date_mutation").str.contains("/"))
      .then(pl.col("date_mutation").str.strptime(pl.Date, "%d/%m/%Y", strict=False))
      .otherwise(pl.col("date_mutation").str.strptime(pl.Date, "%Y-%m-%d", strict=False))
      .alias("date_mutation"),
])
print("✅ Types convertis")

# ============================================================
# ETAPE 1 : COMMUNE
# ============================================================
print("\n--- COMMUNE ---")
Commune = (
    data.select(["code_commune", "nom_commune", "code_departement",
                 "ancien_code_commune", "ancien_nom_commune"])
    .drop_nulls(subset=["code_commune"]).unique(subset=["code_commune"])
)
existing = get_existing_keys(conn, "COMMUNE", "code_commune")
Commune = Commune.filter(~pl.col("code_commune").is_in(existing))
print(f"  {len(Commune):,} nouvelles communes")
bulk_insert(conn, "COMMUNE", Commune)
del Commune; gc.collect()

commune_db = load_table(conn,
    "SELECT id_commune, code_commune FROM COMMUNE",
    {"id_commune": pl.Int64, "code_commune": pl.String})
commune_map = {row[1]: row[0] for row in commune_db.iter_rows()}
print(f"  Mapping : {len(commune_map):,} entrées")

# ============================================================
# ETAPE 2 : ADRESSE
# Normalisation des colonnes pour maximiser le matching
# ============================================================
print("\n--- ADRESSE ---")
join_cols_adresse = ["adresse_numero", "adresse_suffixe", "adresse_nom_voie",
                     "adresse_code_voie", "code_postal"]

Adresse = (
    data.select(["adresse_numero", "adresse_suffixe", "adresse_code_voie",
                 "adresse_nom_voie", "code_postal", "longitude", "latitude", "code_commune"])
    .unique(subset=join_cols_adresse)
    .with_columns(
        pl.col("code_commune").replace_strict(
            list(commune_map.keys()), list(commune_map.values()), default=None
        ).cast(pl.Int64).alias("id_commune")
    )
    .select(["adresse_numero", "adresse_suffixe", "adresse_code_voie",
             "adresse_nom_voie", "code_postal", "longitude", "latitude", "id_commune"])
)
if count_rows(conn, "ADRESSE") == 0:
    print(f"  {len(Adresse):,} adresses à insérer")
    bulk_insert(conn, "ADRESSE", Adresse)
else:
    print("  ⚠️  ADRESSE déjà remplie — ignorée")
del Adresse; gc.collect()

# Charge le mapping adresse avec normalisation
adresse_db = load_table(conn,
    "SELECT id_adresse, adresse_numero, adresse_suffixe, adresse_nom_voie, adresse_code_voie, code_postal FROM ADRESSE",
    {"id_adresse": pl.Int64, "adresse_numero": pl.String, "adresse_suffixe": pl.String,
     "adresse_nom_voie": pl.String, "adresse_code_voie": pl.String, "code_postal": pl.String})

# NORMALISATION — clé du succès pour le matching des adresses
data = normalize_cols(data, join_cols_adresse)
adresse_db = normalize_cols(adresse_db, join_cols_adresse)

data = data.join(adresse_db, on=join_cols_adresse, how="left")
print("  id_adresse appliqué")
del adresse_db; gc.collect()

# ============================================================
# ETAPE 3 : NATURE_CULTURE
# ============================================================
print("\n--- NATURE_CULTURE ---")
NatureCulture = (
    data.select(["code_nature_culture", "nature_culture"])
    .drop_nulls(subset=["code_nature_culture"]).unique()
)
existing = get_existing_keys(conn, "Nature_culture", "code_nature_culture")
NatureCulture = NatureCulture.filter(~pl.col("code_nature_culture").is_in(existing))
print(f"  {len(NatureCulture):,} nouvelles")
bulk_insert(conn, "Nature_culture", NatureCulture)
del NatureCulture; gc.collect()

nc_db = load_table(conn,
    "SELECT id_nature_culture, code_nature_culture FROM Nature_culture",
    {"id_nature_culture": pl.Int64, "code_nature_culture": pl.String})

# ============================================================
# ETAPE 4 : NATURE_CULTURE_SPECIALE
# ============================================================
print("\n--- NATURE_CULTURE_SPECIALE ---")
NatureCultureSpe = (
    data.select(["code_nature_culture_speciale", "nature_culture_speciale"])
    .drop_nulls(subset=["code_nature_culture_speciale"]).unique()
)
existing = get_existing_keys(conn, "Nature_culture_speciale", "code_nature_culture_speciale")
NatureCultureSpe = NatureCultureSpe.filter(~pl.col("code_nature_culture_speciale").is_in(existing))
print(f"  {len(NatureCultureSpe):,} nouvelles")
bulk_insert(conn, "Nature_culture_speciale", NatureCultureSpe)
del NatureCultureSpe; gc.collect()

ncs_db = load_table(conn,
    "SELECT id_nature_culture_speciale, code_nature_culture_speciale FROM Nature_culture_speciale",
    {"id_nature_culture_speciale": pl.Int64, "code_nature_culture_speciale": pl.String})

# ============================================================
# ETAPE 5 : BIEN — 4 types seulement
# surface_reelle_bati et nombre_pieces_principales sont dans Mutation_bien
# ============================================================
print("\n--- BIEN ---")
Bien = (
    data.select(["code_type_local", "type_local"])
    .drop_nulls(subset=["code_type_local"])
    .unique()
)
if count_rows(conn, "BIEN") == 0:
    print(f"  {len(Bien):,} types de biens à insérer")
    bulk_insert(conn, "BIEN", Bien)
else:
    print("  ⚠️  BIEN déjà remplie — ignorée")
del Bien; gc.collect()

# Charge mapping BIEN par code_type_local
bien_db = load_table(conn,
    "SELECT id_bien, code_type_local FROM BIEN",
    {"id_bien": pl.Int64, "code_type_local": pl.String})
data = data.join(bien_db, on="code_type_local", how="left")
print("  id_bien appliqué")
del bien_db; gc.collect()

# ============================================================
# ETAPE 6 : PARCELLE
# id_adresse vient directement de data (join étape 2 avec normalisation)
# surface_terrain ajoutée depuis le CSV
# ============================================================
print("\n--- PARCELLE ---")
Parcelle = (
    data.select(["id_parcelle", "ancien_id_parcelle",
                 "lot1_surface_carrez", "lot2_surface_carrez",
                 "lot3_surface_carrez", "lot4_surface_carrez", "lot5_surface_carrez",
                 "lot1_numero", "lot2_numero", "lot3_numero", "lot4_numero", "lot5_numero",
                 "numero_disposition", "numero_volume", "surface_terrain", "id_adresse"])
    .rename({
        "lot1_surface_carrez": "lot_1_surface_terrain",
        "lot2_surface_carrez": "lot_2_surface_terrain",
        "lot3_surface_carrez": "lot_3_surface_terrain",
        "lot4_surface_carrez": "lot_4_surface_terrain",
        "lot5_surface_carrez": "lot_5_surface_terrain",
        "lot1_numero": "lot_1_numero",
        "lot2_numero": "lot_2_numero",
        "lot3_numero": "lot_3_numero",
        "lot4_numero": "lot_4_numero",
        "lot5_numero": "lot_5_numero",
    })
    .drop_nulls(subset=["id_parcelle"]).unique(subset=["id_parcelle"])
)
existing = get_existing_keys(conn, "PARCELLE", "id_parcelle")
Parcelle = Parcelle.filter(~pl.col("id_parcelle").is_in(existing))
print(f"  {len(Parcelle):,} nouvelles parcelles")
bulk_insert(conn, "PARCELLE", Parcelle)
del Parcelle; gc.collect()

parcelle_db = load_table(conn,
    "SELECT n_parcelle, id_parcelle FROM PARCELLE",
    {"n_parcelle": pl.Int64, "id_parcelle": pl.String})
data = data.join(parcelle_db, on="id_parcelle", how="left")
print("  n_parcelle appliqué")
del parcelle_db; gc.collect()

# ============================================================
# ETAPE 7 : MUTATION
# n_parcelle et date_mutation directement depuis data
# ============================================================
print("\n--- MUTATION ---")
Mutation = (
    data.select(["id_mutation", "valeur_fonciere", "nature_mutation",
                 "date_mutation", "n_parcelle"])
    .rename({"nature_mutation": "nature_vente"})
    .drop_nulls(subset=["id_mutation"])
    .unique(subset=["id_mutation"])
)
existing = get_existing_keys(conn, "MUTATION", "id_mutation")
Mutation = Mutation.filter(~pl.col("id_mutation").is_in(existing))
print(f"  {len(Mutation):,} nouvelles mutations")
bulk_insert(conn, "MUTATION", Mutation)
del Mutation; gc.collect()

mutation_db = load_table(conn,
    "SELECT n_mutation, id_mutation FROM MUTATION",
    {"n_mutation": pl.Int64, "id_mutation": pl.String})
data = data.join(mutation_db, on="id_mutation", how="left")
print("  n_mutation appliqué")
del mutation_db; gc.collect()

# ============================================================
# ETAPE 8 : MUTATION_BIEN
# Contient surface_reelle_bati et nombre_pieces_principales
# ============================================================
print("\n--- MUTATION_BIEN ---")
MutationBien = (
    data.select(["n_mutation", "id_bien", "date_mutation",
                 "surface_reelle_bati", "nombre_pieces_principales"])
    .drop_nulls(subset=["n_mutation", "id_bien", "date_mutation"])
    .unique()
)
if count_rows(conn, "Mutation_bien") == 0:
    print(f"  {len(MutationBien):,} lignes à insérer")
    bulk_insert(conn, "Mutation_bien", MutationBien)
else:
    print("  ⚠️  Mutation_bien déjà remplie — ignorée")
del MutationBien; gc.collect()

# ============================================================
# ETAPE 9 : BIEN_PARCELLE
# ============================================================
print("\n--- BIEN_PARCELLE ---")
BienParcelle = (
    data.select(["id_bien", "n_parcelle"])
    .drop_nulls().unique()
)
if count_rows(conn, "Bien_parcelle") == 0:
    print(f"  {len(BienParcelle):,} lignes à insérer")
    bulk_insert(conn, "Bien_parcelle", BienParcelle)
else:
    print("  ⚠️  Bien_parcelle déjà remplie — ignorée")
del BienParcelle; gc.collect()

# ============================================================
# ETAPE 10 : PARCELLE_NAT_CULT
# ============================================================
print("\n--- PARCELLE_NAT_CULT ---")
ParcelleNatCult = (
    data.select(["n_parcelle", "code_nature_culture"]).drop_nulls().unique()
    .join(nc_db, on="code_nature_culture", how="left")
    .select(["n_parcelle", "id_nature_culture"]).drop_nulls().unique()
)
if count_rows(conn, "Parcelle_nat_cult") == 0:
    print(f"  {len(ParcelleNatCult):,} lignes à insérer")
    bulk_insert(conn, "Parcelle_nat_cult", ParcelleNatCult)
else:
    print("  ⚠️  Parcelle_nat_cult déjà remplie — ignorée")
del ParcelleNatCult; gc.collect()

# ============================================================
# ETAPE 11 : PARCELLE_NAT_CULT_SPE
# ============================================================
print("\n--- PARCELLE_NAT_CULT_SPE ---")
ParcelleNatCultSpe = (
    data.select(["n_parcelle", "code_nature_culture_speciale"]).drop_nulls().unique()
    .join(ncs_db, on="code_nature_culture_speciale", how="left")
    .select(["n_parcelle", "id_nature_culture_speciale"]).drop_nulls().unique()
)
if count_rows(conn, "Parcelle_nat_cult_spe") == 0:
    print(f"  {len(ParcelleNatCultSpe):,} lignes à insérer")
    bulk_insert(conn, "Parcelle_nat_cult_spe", ParcelleNatCultSpe)
else:
    print("  ⚠️  Parcelle_nat_cult_spe déjà remplie — ignorée")
del ParcelleNatCultSpe; gc.collect()

conn.close()
print("\n🎉 Intégration complète terminée !")

✅ Connexion établie

📂 Lecture du CSV complet en mémoire...
✅ CSV chargé : 20,102,739 lignes
✅ Types convertis

--- COMMUNE ---
  33,380 nouvelles communes
  33,380 / 33,380 lignes insérées dans COMMUNE
  Mapping : 33,380 entrées

--- ADRESSE ---
  5,262,304 adresses à insérer
  5,262,304 / 5,262,304 lignes insérées dans ADRESSE
